# Using GRIB data

In [1]:
import earthkit.data as ekd

ekd.download_example_file("test6.grib")

In [2]:
ds_in = ekd.from_source("file", "test6.grib")
ds_in

path,test6.grib
size,1.4 KiB
types,"fieldlist, pandas, xarray, numpy, array"


We load the GRIB data into a fieldlist.

In [3]:
ds = ds_in.to_fieldlist()

### Iteration

In [4]:
for f in ds:
    print(f)

Field(t, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 1000, pressure, 0, regular_ll)
Field(u, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 1000, pressure, 0, regular_ll)
Field(v, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 1000, pressure, 0, regular_ll)
Field(t, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 850, pressure, 0, regular_ll)
Field(u, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 850, pressure, 0, regular_ll)
Field(v, 2018-08-01 12:00:00, 2018-08-01 12:00:00, 0:00:00, 850, pressure, 0, regular_ll)


### Inspecting the contents

In [5]:
len(ds)

6

In [6]:
ds.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
1,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
2,v,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
3,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll
4,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll
5,v,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll


### Slicing

Standard Python slicing is available. It does not involve any loading/copying of GRIB data. 

In [7]:
g = ds[1]
g.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll


In [8]:
g = ds[1:3]
g.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
1,v,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll


In [9]:
g = ds[-1]
g.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,v,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll


### Getting data values

#### Using values

In [10]:
v = ds[0].values
v.shape

(84,)

In [11]:
v[0:4]

array([272.56417847, 272.56417847, 272.56417847, 272.56417847])

In [12]:
v = ds.values
v.shape

(6, 84)

#### Using to_numpy()

In [13]:
v = ds[0].to_numpy()
v.shape

(7, 12)

In [14]:
v = ds.to_numpy()
v.shape

(6, 7, 12)

### Fields and metadata

A fieldlist is made up of fields. We can use the automatic display to get a quick overview about what a field contains.

In [15]:
ds[0]

number_of_values,84
array_type,ndarray
array_dtype,float64
variable,t
units,kelvin
valid_datetime,2018-08-01 12:00:00
base_datetime,2018-08-01 12:00:00
step,0:00:00
level,1000
layer,None
level_type,pressure


In [16]:
ds[0].get("vertical.level")

1000

In [17]:
ds[0:2].get(["vertical.level", "parameter.variable"])

[[1000, 't'], [1000, 'u']]

In [18]:
ds.get("vertical.level")

[1000, 1000, 1000, 850, 850, 850]

The underlying ecCodes GRIB metadata keys can also be accessed.

In [19]:
ds[0:2].metadata(["level", "shortName", "centre"])

[[1000, 't', 'ecmf'], [1000, 'u', 'ecmf']]

### Selection

In [20]:
g = ds.sel({"parameter.variable": ["u", "v"], "vertical.level": 850})
g.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,u,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll
1,v,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll


In [21]:
g = ds.sel({"parameter.variable": "t"})
g.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,1000,pressure,0,regular_ll
1,t,2018-08-01 12:00:00,2018-08-01 12:00:00,0 days,850,pressure,0,regular_ll


### Xarray

In [22]:
ds.to_xarray()

<xarray.Dataset> Size: 4kB
Dimensions:    (level: 2, latitude: 7, longitude: 12)
Coordinates:
  * level      (level) int64 16B 850 1000
  * latitude   (latitude) float64 56B 90.0 60.0 30.0 0.0 -30.0 -60.0 -90.0
  * longitude  (longitude) float64 96B 0.0 30.0 60.0 90.0 ... 270.0 300.0 330.0
Data variables:
    t          (level, latitude, longitude) float64 1kB ...
    u          (level, latitude, longitude) float64 1kB ...
    v          (level, latitude, longitude) float64 1kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF